## Cleaning - Eurostat long term residents Cameroon

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path("..")
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

EUROPE_PATH = DATA_RAW / "europe"
CLEANED_PATH = DATA_PROCESSED / "cleaned"

CLEANED_PATH.mkdir(parents=True, exist_ok=True)

In [2]:
xls = pd.ExcelFile(EUROPE_PATH / "eurostat_status_changes_cameroon.xlsx")
xls.sheet_names

/home/linno/Documents/My project/cameroon-migration-analysis/.venv/lib/python3.12/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


['Summary',
 'Sheet 1',
 'Sheet 2',
 'Sheet 3',
 'Sheet 4',
 'Sheet 5',
 'Sheet 6',
 'Sheet 7',
 'Sheet 8',
 'Sheet 9',
 'Sheet 10',
 'Sheet 11',
 'Sheet 12',
 'Sheet 13',
 'Sheet 14',
 'Sheet 15',
 'Sheet 16',
 'Sheet 17',
 'Sheet 18',
 'Sheet 19',
 'Sheet 20',
 'Sheet 21',
 'Sheet 22',
 'Sheet 23',
 'Sheet 24',
 'Sheet 25',
 'Sheet 26',
 'Sheet 27',
 'Sheet 28',
 'Sheet 29',
 'Sheet 30',
 'Sheet 31',
 'Sheet 32',
 'Sheet 33',
 'Sheet 34',
 'Sheet 35',
 'Sheet 36',
 'Sheet 37',
 'Sheet 38',
 'Sheet 39',
 'Sheet 40',
 'Sheet 41',
 'Sheet 42',
 'Sheet 43',
 'Sheet 44',
 'Sheet 45',
 'Sheet 46',
 'Sheet 47',
 'Sheet 48',
 'Sheet 49',
 'Sheet 50',
 'Sheet 51',
 'Sheet 52',
 'Sheet 53',
 'Sheet 54',
 'Sheet 55',
 'Sheet 56',
 'Sheet 57',
 'Sheet 58',
 'Sheet 59',
 'Sheet 60',
 'Sheet 61',
 'Sheet 62',
 'Sheet 63',
 'Sheet 64',
 'Sheet 65',
 'Sheet 66',
 'Sheet 67',
 'Sheet 68',
 'Sheet 69',
 'Sheet 70',
 'Sheet 71',
 'Sheet 72',
 'Sheet 73',
 'Sheet 74',
 'Sheet 75',
 'Sheet 76',
 'Sheet 7

In [3]:
sheet1_raw = pd.read_excel(
    EUROPE_PATH / "eurostat_status_changes_cameroon.xlsx",
    sheet_name="Sheet 1",
    header=None
)

sheet1_raw

,0,1,2,3,4,5,6,7,8,9,10
0,Data extracted on 09/04/2026 14:05:58 from [ES...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Dataset:,"Change of immigration status permits by age, s...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Last updated:,23/03/2026 23:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Time frequency,NaN,Annual,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Country of citizenship,NaN,Cameroon,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,Age class,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,Sex,NaN,Total,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Unit of measure,NaN,Person,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
sheet1_raw = sheet1_raw.drop(columns=[2, 4, 6, 8, 10], errors="ignore")
sheet1_raw = sheet1_raw.drop(index=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 44, 45, 46, 47, 48], errors="ignore").reset_index(drop=True)

sheet1_raw

,0,1,3,5,7,9
0,TIME,2020,2021,2022,2023,2024.0
1,GEO (Labels),NaN,NaN,NaN,NaN,NaN
2,European Union - 27 countries (from 2020),:,:,:,:,6696.0
3,Belgium,:,426,504,656,548.0
4,Bulgaria,:,0,1,0,1.0
5,Czechia,:,3,8,16,10.0
6,Denmark,:,14,35,14,10.0
7,Germany,:,236,517,585,496.0
8,Estonia,:,15,13,4,5.0
9,Ireland,:,9,67,14,24.0


In [5]:
sheet1_raw = sheet1_raw.replace(":", 0)
sheet1_raw = sheet1_raw.rename(
    columns={"TIME": "destination_country"}
)
sheet1_raw = sheet1_raw.drop(index=[1, 2])

sheet1_raw

,0,1,3,5,7,9
0,TIME,2020,2021,2022,2023,2024.0
3,Belgium,0,426,504,656,548.0
4,Bulgaria,0,0,1,0,1.0
5,Czechia,0,3,8,16,10.0
6,Denmark,0,14,35,14,10.0
7,Germany,0,236,517,585,496.0
8,Estonia,0,15,13,4,5.0
9,Ireland,0,9,67,14,24.0
10,Greece,0,6,1,5,3.0
11,Spain,0,109,145,170,138.0


In [6]:
# Mettre la première ligne comme entête
sheet1_raw.columns = sheet1_raw.iloc[0]
sheet1_raw = sheet1_raw.iloc[1:].reset_index(drop=True)

# Renommer TIME en destination_country
sheet1_raw = sheet1_raw.rename(
    columns={"TIME": "destination_country"}
)

# Supprimer la ligne GEO (Labels) si elle existe encore
sheet1_raw = sheet1_raw[
    sheet1_raw["destination_country"] != "GEO (Labels)"
]

# Passer du format large au format long
sheet1_raw = sheet1_raw.melt(
    id_vars=["destination_country"],
    var_name="year",
    value_name="permit"
)

# Nettoyer les types
sheet1_raw["year"] = sheet1_raw["year"].astype(int)

sheet1_raw["permit"] = (
    pd.to_numeric(sheet1_raw["permit"], errors="coerce")
    .fillna(0)
    .astype(int)
)

# Réordonner les colonnes
sheet1_raw = sheet1_raw[
    ["destination_country", "year", "permit"]
]

sheet1_raw


,destination_country,year,permit
0,Belgium,2020,0
1,Bulgaria,2020,0
2,Czechia,2020,0
3,Denmark,2020,0
4,Germany,2020,0
...,...,...,...
150,Sweden,2024,19
151,Iceland,2024,1
152,Liechtenstein,2024,1
153,Norway,2024,4


In [7]:
# Renommer permit en permits
sheet1_raw = sheet1_raw.rename(
    columns={"permit": "permits"}
)

In [8]:
# Ajouter les colonnes source et nature
sheet1_raw["source"] = "eurostat"
sheet1_raw["nature"] = "status_change"

# Ajouter la période Covid
def classify_covid_period(year):
    if year < 2020:
        return "pre_covid"
    elif year in [2020, 2021]:
        return "covid"
    else:
        return "post_covid"

sheet1_raw["period_covid"] = sheet1_raw["year"].apply(classify_covid_period)

# Réordonner les colonnes
sheet1_raw = sheet1_raw[
    [
        "destination_country",
        "year",
        "permits",
        "source",
        "nature",
        "period_covid"
    ]
]

sheet1_raw


,destination_country,year,permits,source,nature,period_covid
0,Belgium,2020,0,eurostat,status_change,covid
1,Bulgaria,2020,0,eurostat,status_change,covid
2,Czechia,2020,0,eurostat,status_change,covid
3,Denmark,2020,0,eurostat,status_change,covid
4,Germany,2020,0,eurostat,status_change,covid
...,...,...,...,...,...,...
150,Sweden,2024,19,eurostat,status_change,post_covid
151,Iceland,2024,1,eurostat,status_change,post_covid
152,Liechtenstein,2024,1,eurostat,status_change,post_covid
153,Norway,2024,4,eurostat,status_change,post_covid


In [9]:
sheet1_raw = sheet1_raw.rename(
    columns={"period_covid": "covid_period"}
)

sheet1_raw

,destination_country,year,permits,source,nature,covid_period
0,Belgium,2020,0,eurostat,status_change,covid
1,Bulgaria,2020,0,eurostat,status_change,covid
2,Czechia,2020,0,eurostat,status_change,covid
3,Denmark,2020,0,eurostat,status_change,covid
4,Germany,2020,0,eurostat,status_change,covid
...,...,...,...,...,...,...
150,Sweden,2024,19,eurostat,status_change,post_covid
151,Iceland,2024,1,eurostat,status_change,post_covid
152,Liechtenstein,2024,1,eurostat,status_change,post_covid
153,Norway,2024,4,eurostat,status_change,post_covid


In [10]:
sheet1_raw.to_csv(
    CLEANED_PATH / "eurostat_status_changes_cleaned.csv",
    index=False
)